In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
# Load datasets
tts = pd.read_csv("../data/TTS_LBNL_public_file_29-Sep-2025_all.csv", low_memory=False)
dg = pd.read_csv("../data/PGE_Interconnected_Project_Sites_Jan2020-Jan2026.csv", low_memory=False)

print(f"TTS: {len(tts):,} rows, {tts.shape[1]} columns")
print(f"DG Stats: {len(dg):,} rows, {dg.shape[1]} columns")

## 1. Filter TTS to PG&E California

In [19]:
# Filter TTS to CA / PG&E territory, residential only
RES_SEGMENTS = ['RES', 'RES_SF', 'RES_MF']

tts['installation_date'] = pd.to_datetime(tts['installation_date'], errors='coerce')

tts_filter = tts[
    (tts['state'] == 'CA') &
    (tts['utility_service_territory'].str.contains('Pacific Gas', case=False, na=False)) &
    (tts['customer_segment'].isin(RES_SEGMENTS)) &
    (tts['installation_date'].dt.year.between(2020, 2024))
].copy()

print(f"TTS PG&E CA residential (2020-2024): {len(tts_filter):,} rows")

dg['App Approved Date'] = pd.to_datetime(dg['App Approved Date'], errors='coerce')

dg_filter = dg[
    (dg['Customer Sector'] == 'Residential') &
    (dg['App Approved Date'].dt.year.between(2020, 2024))
].copy()

print(f"DG Stats PG&E 20-2024 residential: {len(dg_filter):,} rows")

TTS PG&E CA residential (2020-2024): 479,988 rows
DG Stats PG&E 20-2024 residential: 465,968 rows


## 2. Schema Comparison — Key Fields

In [21]:
# Align key fields to a common schema — residential only
tts_key = tts_filter[['installation_date', 'PV_system_size_DC', 'total_installed_price',
                    'customer_segment', 'zip_code', 'installer_name',
                    'third_party_owned', 'battery_rated_capacity_kWh']].copy()
tts_key.columns = ['date', 'system_size_dc_kw', 'total_cost',
                   'customer_segment', 'zip', 'installer',
                   'third_party_owned', 'battery_kwh']
tts_key['source'] = 'TTS'


dg_key = dg_filter[['App Approved Date', 'System Size DC', 'Total System Cost',
                  'Customer Sector', 'Service Zip', 'Installer Name',
                  'Third Party Owned', 'Storage Capacity (kWh)']].copy()
dg_key.columns = ['date', 'system_size_dc_kw', 'total_cost',
                  'customer_segment', 'zip', 'installer',
                  'third_party_owned', 'battery_kwh']
dg_key['source'] = 'DG Stats'

print(f"TTS residential: {len(tts_key):,} rows")

print("\nTTS sample:")
print(tts_key.head(3))

print(f"DG residential: {len(dg_key):,} rows")

print("\nDG Stats sample:")
print(dg_key.head(3))

TTS residential: 479,988 rows

TTS sample:
              date  system_size_dc_kw  total_cost customer_segment      zip  \
1654106 2024-12-30               5.21   23,296.00           RES_SF  95618.0   
1654108 2024-12-30               9.43   19,000.00           RES_SF  95060.0   
1654109 2024-12-30               9.02   18,758.00           RES_SF  95060.0   

                installer  third_party_owned  battery_kwh source  
1654106  West Coast Solar               0.00        -1.00    TTS  
1654108              Self               0.00        -1.00    TTS  
1654109              Self               0.00        -1.00    TTS  
DG residential: 465,968 rows

DG Stats sample:
         date  system_size_dc_kw  total_cost customer_segment       zip  \
21 2023-02-02               3.84         NaN      Residential 95,746.00   
22 2023-02-28              11.30        0.00      Residential 93,611.00   
25 2021-12-09              15.60   49,500.00      Residential 95,231.00   

                        

## 3. Distribution Comparison — System Size & Cost

In [22]:
for col, label in [('system_size_dc_kw', 'System Size DC (kW)'), ('total_cost', 'Total Cost ($)')]:
    tts_vals = pd.to_numeric(tts_key[col], errors='coerce').dropna()
    dg_vals = pd.to_numeric(dg_key[col], errors='coerce').dropna()
    print(f"\n--- {label} ---")
    print(f"{'':20s} {'TTS':>12s} {'DG Stats':>12s}")
    print(f"{'count':20s} {len(tts_vals):>12,.0f} {len(dg_vals):>12,.0f}")
    print(f"{'median':20s} {tts_vals.median():>12,.1f} {dg_vals.median():>12,.1f}")
    print(f"{'mean':20s} {tts_vals.mean():>12,.1f} {dg_vals.mean():>12,.1f}")
    print(f"{'p25':20s} {tts_vals.quantile(.25):>12,.1f} {dg_vals.quantile(.25):>12,.1f}")
    print(f"{'p75':20s} {tts_vals.quantile(.75):>12,.1f} {dg_vals.quantile(.75):>12,.1f}")


--- System Size DC (kW) ---
                              TTS     DG Stats
count                     479,988      465,968
median                        6.0          6.3
mean                          7.0          7.1
p25                           4.2          4.4
p75                           8.5          8.8

--- Total Cost ($) ---
                              TTS     DG Stats
count                     479,988      463,515
median                   25,167.0     21,450.0
mean                     30,419.9     24,455.1
p25                      16,200.0          0.0
p75                      37,985.0     35,802.0


## 3b. Categorical & Date Distribution Comparisons

In [27]:
# --- Year of install ---
tts_year = pd.to_datetime(tts_key['date'], errors='coerce').dt.year.value_counts().sort_index()
dg_year = pd.to_datetime(dg_key['date'], errors='coerce').dt.year.value_counts().sort_index()

year_df = pd.DataFrame({'TTS': tts_year, 'DG Stats': dg_year}).fillna(0).astype(int)
print("=== Install Year ===")
print(year_df.to_string())

# --- Application status (DG Stats only) ---
print("\n=== Application Status (DG Stats only) ===")
print(dg_filter['Application Status'].value_counts())


=== Install Year ===
         TTS  DG Stats
date                  
2020   70607     64712
2021   88359     83718
2022  119704    116467
2023  124630    124115
2024   76688     76956

=== Application Status (DG Stats only) ===
Application Status
Interconnected    465968
Name: count, dtype: int64


In [28]:
# --- Mounting method ---
# TTS: ground_mounted (1.0 = ground, 0.0 = roof)
# DG Stats: Mounting Method (categorical)
print("\n=== Mounting Method ===")
tts_mount = tts_filter['ground_mounted'].map({1.0: 'Ground', 0.0: 'Roof/Other', -1.0: 'Unknown'}).value_counts()
print("TTS:")
print(tts_mount)
print(f"\nTTS % ground: {(tts_filter['ground_mounted'] == 1.0).mean():.1%}")

print("\nDG Stats:")
print(dg_filter['Mounting Method'].value_counts(dropna=False))



=== Mounting Method ===
TTS:
ground_mounted
Roof/Other    455127
Unknown        15655
Ground          9206
Name: count, dtype: int64

TTS % ground: 1.9%

DG Stats:
Mounting Method
Rooftop     449150
Ground        9567
NaN           6603
Mixed          506
multiple       142
Name: count, dtype: int64


In [29]:

# --- Third party owned ---
print("\n=== Third Party Owned ===")
tts_tpo = tts_filter['third_party_owned'].value_counts(dropna=False)
print("TTS (raw values):")
print(tts_tpo)

dg_tpo = dg_filter['Third Party Owned'].value_counts(dropna=False)
print("\nDG Stats:")
print(dg_tpo)


=== Third Party Owned ===
TTS (raw values):
third_party_owned
0.00     357976
1.00     121705
-1.00       307
Name: count, dtype: int64

DG Stats:
Third Party Owned
No     349807
Yes    116161
Name: count, dtype: int64


## 4. Project Matching — Find Overlapping Records

In [ ]:
# Build match keys: zip + system size (rounded to 2dp) + year
def make_match_key(df):
    d = df.copy()
    d['date'] = pd.to_datetime(d['date'], errors='coerce')
    d['year'] = d['date'].dt.year
    d['size_rounded'] = pd.to_numeric(d['system_size_dc_kw'], errors='coerce').round(2)
    d['zip_clean'] = d['zip'].astype(str).str.strip().str[:5]
    d['match_key'] = d['zip_clean'] + '_' + d['size_rounded'].astype(str) + '_' + d['year'].astype(str)
    return d

tts_m = make_match_key(tts_key)
dg_m = make_match_key(dg_key)

tts_keys = set(tts_m['match_key'].dropna())
dg_keys = set(dg_m['match_key'].dropna())
overlap = tts_keys & dg_keys

print(f"TTS unique keys:      {len(tts_keys):,}")
print(f"DG Stats unique keys: {len(dg_keys):,}")
print(f"Overlapping keys:     {len(overlap):,}")
print(f"Match rate (vs DG):   {len(overlap)/len(dg_keys):.1%}")

In [ ]:
# Inspect matched records side by side
matched_tts = tts_m[tts_m['match_key'].isin(overlap)]
matched_dg = dg_m[dg_m['match_key'].isin(overlap)]

print(f"Matched TTS rows: {len(matched_tts):,}")
print(f"Matched DG rows:  {len(matched_dg):,}")
matched_dg.head(5)